In [1]:
import langchain
import langgraph
import openai
import importlib
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
print("LangChain版本：", langchain.__version__)
print("LangGraph版本：", importlib.metadata.version("langgraph"))
print("OpenAI版本：", openai.__version__)

LangChain版本： 1.2.15
LangGraph版本： 1.1.6
OpenAI版本： 2.31.0


In [7]:
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")  # API地址，使用你的模型对应的地址（如DeepSeek: https://api.deepseek.com）

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

load_dotenv()

API_KEY = SecretStr(os.getenv('API_KEY')) # type: ignore
BASE_URL = os.getenv("BASE_URL")

if not API_KEY:
    raise ValueError("未检测到 API_KEY")

llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="qwen3.5-plus",  # 替换为你在百炼开通的模型，例如 qwen-plus, qwen-max, deepseek-r1
    temperature=0.9
)

# 构造 Prompt
prompt = "请写一段50字左右的 AI 学习建议，语言简洁、实用，适合初学者。"

# 调用模型
response = llm.invoke(prompt)

# 输出结果
print("生成的学习建议：")
print(response.content)

生成的学习建议：
先学编程数学基础再学框架。重在动手实战，多做项目少空谈。关注前沿多交流。保持好奇，坚持实践，终有成。


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from pydantic import SecretStr  # 可选，用于类型检查

# 加载环境变量
load_dotenv()

API_KEY = SecretStr(os.getenv("API_KEY"))  # type: ignore # 使用 SecretStr 避免类型警告
BASE_URL = os.getenv("BASE_URL")

if not API_KEY:
    raise ValueError("未检测到 API_KEY，请检查 .env 文件")

# 初始化大模型（使用阿里云百炼的模型名）
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="qwen-plus",  # 换成你在百炼开通的模型，如 qwen-plus, qwen-max, deepseek-r1
    temperature=0.3
)

# 定义 State，增加 english_advice 字段
class WorkflowState(TypedDict, total=False):
    user_role: str
    original_advice: str
    simplified_advice: str
    english_advice: str   # 新增字段

# 节点1：生成原始建议
def generate_advice(state: WorkflowState):
    prompt = f"给{state['user_role']}写一段50字左右的 AI 学习建议。" # type: ignore
    result = llm.invoke(prompt)
    return {"original_advice": result.content}

# 节点2：精简建议
def simplify_advice(state: WorkflowState):
    prompt = f"把下面的学习建议精简到30字以内：{state['original_advice']}" # type: ignore
    result = llm.invoke(prompt)
    return {"simplified_advice": result.content}

# 节点3：翻译成英语
def translate_to_english(state: WorkflowState):
    prompt = f"请将以下中文文本翻译成英语：{state['simplified_advice']}" # type: ignore
    result = llm.invoke(prompt)
    return {"english_advice": result.content}

# 构建工作流
workflow = StateGraph(WorkflowState)

workflow.add_node("generate", generate_advice)
workflow.add_node("simplify", simplify_advice)
workflow.add_node("translate", translate_to_english)

workflow.add_edge(START, "generate")
workflow.add_edge("generate", "simplify")
workflow.add_edge("simplify", "translate")
workflow.add_edge("translate", END)

app = workflow.compile()

# 执行
result = app.invoke({"user_role": "高校学生"})

# 输出结果
print("原始学习建议：")
print(result["original_advice"])
print("\n精简后学习建议：")
print(result["simplified_advice"])
print("\n翻译成英语：")
print(result["english_advice"])

原始学习建议：
AI学习重在实践：先掌握Python与数学基础，再通过Kaggle、天池等平台做小项目；善用开源模型（如Hugging Face），多读论文、勤写笔记，避免只看不练。

精简后学习建议：
AI学习重实践：打牢Python与数学基础，多做项目、读论文、写笔记。

翻译成英语：
AI Learning Emphasizes Practical Application: Build a Solid Foundation in Python and Mathematics, Undertake Numerous Projects, Read Research Papers, and Maintain Detailed Notes.
